# 📄 Descripción general del código

 Este script evalúa la calidad de las relaciones Claim–Evidence previamente predichas
 utilizando un modelo de clasificación binaria (XLM-Roberta) entrenado previamente.
 El objetivo es calcular una métrica llamada "LogWeightedMatchRate" por respuesta,
 y luego normalizarla para obtener un puntaje comparativo por postulante.

# 🔧 Componentes principales:

 1. Se cargan el modelo y el tokenizador desde Google Drive.
 2. Se lee un archivo Excel que contiene pares de fragmentos clasificados como Claim o Evidence.
 3. Para cada grupo (por applicant_id y answer_id):
     - Se emparejan todos los Claims con todos los Evidences.
     - Se predice si hay una relación Claim–Evidence usando el modelo.
     - Se calcula la cantidad de coincidencias con score > 0.5.
     - Se aplica una transformación logarítmica (log1p) al conteo de matches.
     - Se promedia el valor logarítmico por número de Claims → LogWeightedMatchRate.
 4. Se normalizan los valores calculados a un rango de 0 a 1.
 5. Se calcula el promedio de los valores normalizados por cada applicant_id.
 6. Se exportan dos archivos Excel:
     - Uno con detalles por cada answer_id.
     - Otro con el puntaje promedio por cada applicant_id.

 # 🧠 Aplicación:
 Este pipeline permite medir la fortaleza de la argumentación de un postulante
 en términos de cuántas afirmaciones (Claims) están respaldadas por evidencias concretas.
 Puede utilizarse como parte de un sistema de evaluación automática de solicitudes.

# 📁 Archivos de salida:
 - 2claim_match_detail_by_respuesta.xlsx: Detalle de coincidencias por respuesta.
 - claim_match_rate_by_applicant.xlsx: Puntaje promedio de match por postulante.

In [ ]:
!pip install transformers -q
!pip install openpyxl -q

In [ ]:
# ✅ Pipeline completo modificado (Tasa de coincidencia ponderada por log + normalización)

import pandas as pd
import numpy as np
from math import log1p
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
import torch
from torch.nn.functional import softmax
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✅ Cargar modelo y tokenizador

model_path = "models/support_matching_xlmr"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_path)
model = XLMRobertaForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
# ✅ Cargar datos

df = pd.read_excel("data/predicted_claim_evidence.xlsx")

In [ ]:
# ✅ Función de predicción

def predict_claim_evidence_pairs(pairs, batch_size=16):
    results = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        texts1, texts2 = zip(*batch)
        inputs = tokenizer(
            list(texts1), list(texts2),
            padding=True, truncation=True, return_tensors="pt", max_length=256
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        results.extend(probs)
    return results

In [ ]:
# ✅ Función para calcular tasa de coincidencia ponderada por log

def compute_log_match_rate_per_respuesta(df, model, tokenizer, device, batch_size=16):
    results = []
    grouped = df.groupby(["applicant_id", "answer_id"])

    for (applicant, answer_id), group in tqdm(grouped, desc="🔍 Log-weighted matching"):
        claims = group[group["Final_Label"] == "Claim"]["Fragment"].dropna().astype(str).tolist()
        evidences = group[group["Final_Label"] == "Evidence"]["Fragment"].dropna().astype(str).tolist()

        if not claims or not evidences:
            results.append({
                "applicant_id": applicant,
                "answer_id": answer_id,
                "NumClaims": len(claims),
                "LogWeightedMatchRate": 0.0
            })
            continue

        total_log_support = 0
        for claim in claims:
            pairs = [(claim, ev) for ev in evidences]
            scores = predict_claim_evidence_pairs(pairs)
            support_count = sum(score > 0.5 for score in scores)
            total_log_support += log1p(support_count)

        log_weighted_match_rate = total_log_support / len(claims)
        results.append({
            "applicant_id": applicant,
            "answer_id": answer_id,
            "NumClaims": len(claims),
            "LogWeightedMatchRate": round(log_weighted_match_rate, 4)
        })

    return pd.DataFrame(results)

In [ ]:
# ✅ Ejecutar cálculo

df_match_log = compute_log_match_rate_per_respuesta(df, model, tokenizer, device)

In [ ]:
# ✅ Normalización (entre 0 y 1)

min_rate = df_match_log["LogWeightedMatchRate"].min()
max_rate = df_match_log["LogWeightedMatchRate"].max()
df_match_log["Normalized_MatchRate"] = ((df_match_log["LogWeightedMatchRate"] - min_rate) / (max_rate - min_rate)).round(4)

In [ ]:
# ✅ Calcular promedio por applicant_id

df_final_score = df_match_log.groupby("applicant_id")["Normalized_MatchRate"].mean().reset_index()
df_final_score.rename(columns={"Normalized_MatchRate": "Mean_MatchRate"}, inplace=True)

In [ ]:
# ✅ Guardar resultados

df_final_score.to_excel("data/claim_match_rate_by_applicant.xlsx", index=False)
df_match_log.to_excel("data/claim_match_detail_by_answer.xlsx", index=False)